# Final Testing of Standalone Models (2023–2024): SARIMAX, LSTM and XGBoost

Final out-of-sample test of the standalone year-ahead models on the held-out years 2023 and 2024. **No modelling choice is changed here:**

- **SARIMAX:** each grid uses the orders selected during rolling validation (read from `outputs/standalone_models/models/sarimax_rolling_diagnostics.csv`).
- **LSTM and XGBoost:** same architecture, hyperparameters, early-stopping rule and seed as in rolling validation.
- **Climatology and seasonal naive** are included as reference forecasts.

## Test rounds

| Round | Training | Forecast origin | Test year |
|---|---|---|---|
| 1 | 2005–2022 | Dec 2022 | 2023 (12 monthly forecasts) |
| 2 | 2005–2023 (**retrained**) | Dec 2023 | 2024 (12 monthly forecasts) |

## Pooled error metrics
Both test years are combined into **one** evaluation set per model (93 grids × 24 months = 2,232 forecasts). Metrics are computed on all forecasts at once, **not** as the average of per-year or per-grid values:

- **Pooled RMSE** = √( Σ (ŷ − y)² / N )
- **MAE** = Σ |ŷ − y| / N
- **nRMSE** = pooled RMSE / mean(y) × 100 %
- **Coefficient of determination** R² = 1 − Σ (ŷ − y)² / Σ (y − ȳ)², where ȳ is the mean of all 2,232 observations

## Outputs (`outputs/final_test/`)
- `final_test_metrics.csv`: **one row per model**, combined 2023–2024 (pooled RMSE, MAE, nRMSE, R², skill vs climatology)
- `final_test_metrics_by_grid.csv`: the same metrics per model and grid point, pooled over the 24 test months (for spatial mapping)
- `predictions/<model>_final_test_predictions.csv`: every test forecast

Run `Standalone_Models.ipynb` first (the SARIMAX orders come from it), then run this notebook from the `Philippine Spatial Map` folder. Run the setup cell first.

In [1]:
"""
final_test_standalone.py

Purpose:
    Final testing of the standalone year-ahead GHI models on
    2023 and 2024 with pooled error metrics.

Input:
    data/processed/nasa_power_validated.csv
    data/raw/elevation/grid_elevation.csv
    outputs/standalone_models/models/sarimax_rolling_diagnostics.csv

Output:
    outputs/final_test/
"""

from pathlib import Path
from importlib import metadata
import ast
import platform
import random
import warnings

import numpy as np
import pandas as pd


# ============================================================
# 1. USER SETTINGS
# ============================================================

# ------------------------------------------------------------
# INPUT FILES
# ------------------------------------------------------------

DATA_FILE = Path(
    "data/processed/nasa_power_validated.csv"
)

ELEVATION_FILE = Path(
    "data/raw/elevation/grid_elevation.csv"
)

# SARIMAX orders selected during rolling validation
SARIMAX_ORDERS_FILE = Path(
    "outputs/standalone_models/models/sarimax_rolling_diagnostics.csv"
)


# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

OUTPUT_DIR = Path(
    "outputs/final_test"
)

PREDICTION_DIR = OUTPUT_DIR / "predictions"

MODEL_DIR = OUTPUT_DIR / "models"


# ------------------------------------------------------------
# VARIABLES
# ------------------------------------------------------------

GHI_COL = "ghi_kwh_m2_day"

CLOUD_COL = "cloud_amt_pct"

RH_COL = "rh2m_pct"

VARIABLES = [
    GHI_COL,
    CLOUD_COL,
    RH_COL
]

EXOG_COLS = [
    CLOUD_COL,
    RH_COL
]

STATIC_COLS = [
    "latitude",
    "longitude",
    "elevation_m"
]

CALENDAR_COLS = [
    "month_sin",
    "month_cos"
]


# ------------------------------------------------------------
# DATA PERIOD AND TEST YEARS
# ------------------------------------------------------------

DATA_START = "2005-01-01"

DATA_END = "2024-12-01"

# Each test year is forecast from December of the previous
# year, with the models retrained on all data up to that origin
TEST_YEARS = [2023, 2024]


# ------------------------------------------------------------
# FORECAST SETTINGS (same as rolling validation)
# ------------------------------------------------------------

HORIZON = 12

WINDOW = 12

SEASONAL_PERIOD = 12


# ------------------------------------------------------------
# REPRODUCIBILITY
# ------------------------------------------------------------

SEED = 42


# ============================================================
# 2. CREATE OUTPUT DIRECTORIES
# ============================================================

for directory in [PREDICTION_DIR, MODEL_DIR]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# 3. REPRODUCIBILITY
# ============================================================

def set_seed(seed=SEED):
    """Fix the Python and NumPy random seeds."""

    random.seed(seed)

    np.random.seed(seed)


set_seed()


def library_versions():
    """Versions of every library used, for the paper's reproducibility section."""

    versions = {"python": platform.python_version()}

    for package in ["numpy", "pandas", "statsmodels", "scikit-learn", "torch", "xgboost", "joblib"]:

        try:
            versions[package] = metadata.version(package)
        except metadata.PackageNotFoundError:
            versions[package] = "not installed"

    return versions


VERSIONS = library_versions()

(OUTPUT_DIR / "environment.txt").write_text(
    "\n".join(f"{name}=={version}" for name, version in VERSIONS.items()),
    encoding="utf-8"
)

print("Library versions:")

for name, version in VERSIONS.items():
    print(f"  {name:<13} {version}")


# ============================================================
# 4. LOAD DATA
# ============================================================

print("\nLoading data...")

data = pd.read_csv(
    DATA_FILE,
    parse_dates=["date"]
)

elevation = pd.read_csv(
    ELEVATION_FILE
)[["grid_number", "elevation_m"]]

data = (
    data.merge(elevation, on="grid_number", how="left")
    .sort_values(["grid_number", "date"])
    .reset_index(drop=True)
)

data = data[
    (data["date"] >= DATA_START)
    & (data["date"] <= DATA_END)
].reset_index(drop=True)


# ------------------------------------------------------------
# Fill any values set to NaN during validation
# (linear interpolation within each grid)
# ------------------------------------------------------------

n_missing = int(data[VARIABLES].isna().sum().sum())

if n_missing:

    print(f"Interpolating {n_missing} missing values within each grid")

    data[VARIABLES] = (
        data.groupby("grid_number")[VARIABLES]
        .transform(lambda s: s.interpolate(limit_direction="both"))
    )


# ------------------------------------------------------------
# Cyclical month encoding
# ------------------------------------------------------------

data["month_sin"] = np.sin(2 * np.pi * data["month"] / 12)

data["month_cos"] = np.cos(2 * np.pi * data["month"] / 12)


GRID_INFO = (
    data[["grid_number", "grid_id", "latitude", "longitude"]]
    .drop_duplicates("grid_number")
    .reset_index(drop=True)
)

N_GRIDS = len(GRID_INFO)


# The test years must be complete
months_per_year = data[data["year"].isin(TEST_YEARS)].groupby("year").size()

for year in TEST_YEARS:

    if months_per_year.get(year, 0) != N_GRIDS * 12:

        raise ValueError(
            f"Test year {year} has {months_per_year.get(year, 0)} records "
            f"(expected {N_GRIDS * 12})"
        )

print(
    f"Grids: {N_GRIDS}   "
    f"Months: {data['date'].min():%Y-%m} to {data['date'].max():%Y-%m}   "
    f"Records: {len(data)}"
)


# ============================================================
# 5. TEST ROUNDS
# ============================================================

ROUNDS = pd.DataFrame([
    {
        "round":          number,
        "test_year":      year,
        "train_start":    DATA_START[:7],
        "train_end":      f"{year - 1}-12",
        "train_end_date": pd.Timestamp(f"{year - 1}-12-01"),
        "test_start":     pd.Timestamp(f"{year}-01-01"),
        "test_end":       pd.Timestamp(f"{year}-12-01")
    }
    for number, year in enumerate(TEST_YEARS, start=1)
])

ROUND_COLS = [
    "round",
    "test_year",
    "train_start",
    "train_end"
]

GRID_COLS = [
    "grid_number",
    "grid_id",
    "latitude",
    "longitude"
]

print("\nTest rounds:")

print(
    ROUNDS[ROUND_COLS]
    .assign(forecast_origin=ROUNDS["train_end"])
    .to_string(index=False)
)


# ============================================================
# 6. POOLED ERROR METRICS
# ============================================================

def pooled_metrics(frame, keys):
    """
    Pooled metrics over every forecast in each group:

        RMSE  = sqrt( sum((pred - actual)^2) / N )
        MAE   = sum(|pred - actual|) / N
        nRMSE = RMSE / mean(actual) * 100
        R2    = 1 - sum((pred - actual)^2) / sum((actual - mean(actual))^2)

    All sums run over every grid-month in the group (not averages
    of per-year or per-grid values).
    """

    error = frame["predicted"] - frame["actual"]

    group_mean = frame.groupby(keys)["actual"].transform("mean")

    summary = (
        frame.assign(
            squared_error=error ** 2,
            absolute_error=error.abs(),
            squared_deviation=(frame["actual"] - group_mean) ** 2
        )
        .groupby(keys, sort=False)
        .agg(
            SSE=("squared_error", "sum"),
            SAE=("absolute_error", "sum"),
            SST=("squared_deviation", "sum"),
            mean_actual=("actual", "mean"),
            mean_predicted=("predicted", "mean"),
            n_forecasts=("actual", "size")
        )
        .reset_index()
    )

    summary["RMSE"] = np.sqrt(summary["SSE"] / summary["n_forecasts"])

    summary["MAE"] = summary["SAE"] / summary["n_forecasts"]

    summary["nRMSE_pct"] = summary["RMSE"] / summary["mean_actual"] * 100

    summary["R2"] = 1 - summary["SSE"] / summary["SST"]

    return summary[
        keys
        + ["RMSE", "MAE", "nRMSE_pct", "R2", "mean_actual", "mean_predicted", "n_forecasts"]
    ]


# ============================================================
# 7. OUTPUT HELPER
# ============================================================

def save_test_results(predictions, name, label):
    """
    Attach actuals, round and grid information to a model's test
    forecasts and save them. Returns the combined 2023-2024
    frame of forecasts.

    `predictions` needs the columns: round, grid_number,
    target_date, predicted.
    """

    frame = (
        predictions[["round", "grid_number", "target_date", "predicted"]]
        .merge(ROUNDS[ROUND_COLS], on="round", how="left")
        .merge(
            data[["grid_number", "date", GHI_COL]]
            .rename(columns={"date": "target_date", GHI_COL: "actual"}),
            on=["grid_number", "target_date"],
            how="left"
        )
        .merge(GRID_INFO, on="grid_number", how="left")
    )

    frame["model"] = label

    frame["horizon_months"] = frame["target_date"].dt.month

    frame["error"] = frame["predicted"] - frame["actual"]


    expected_rows = len(ROUNDS) * N_GRIDS * HORIZON

    wrong_year = frame["target_date"].dt.year != frame["test_year"]

    if (
        len(frame) != expected_rows
        or frame[["actual", "predicted"]].isna().any().any()
        or wrong_year.any()
    ):

        raise ValueError(
            f"{label}: {len(frame)} forecasts (expected {expected_rows}), "
            f"{frame['predicted'].isna().sum()} missing predictions, "
            f"{wrong_year.sum()} outside their test year"
        )


    frame = frame[
        ["model"] + ROUND_COLS + GRID_COLS
        + ["target_date", "horizon_months", "actual", "predicted", "error"]
    ].sort_values(["round", "grid_number", "target_date"]).reset_index(drop=True)

    path = PREDICTION_DIR / f"{name}_final_test_predictions.csv"

    frame.to_csv(path, index=False)


    overall = pooled_metrics(frame, ["model"]).iloc[0]

    print(
        f"\n{label} final test ({TEST_YEARS[0]}-{TEST_YEARS[-1]} pooled, "
        f"{int(overall['n_forecasts'])} forecasts): "
        f"RMSE {overall['RMSE']:.4f}   MAE {overall['MAE']:.4f}   "
        f"nRMSE {overall['nRMSE_pct']:.2f}%   R² {overall['R2']:.4f}"
    )

    print(f"Saved: {path}")

    return frame

Library versions:
  python        3.14.7
  numpy         2.5.3
  pandas        3.0.6
  statsmodels   0.15.0
  scikit-learn  1.9.1
  torch         2.14.0
  xgboost       3.4.1
  joblib        1.6.0

Loading data...
Grids: 93   Months: 2005-01 to 2024-12   Records: 22320

Test rounds:
 round  test_year train_start train_end forecast_origin
     1       2023     2005-01   2022-12         2022-12
     2       2024     2005-01   2023-12         2023-12


## 1. SARIMAX

Per-grid SARIMAX with cloud amount and RH lagged 12 months as exogenous regressors, using **the orders selected during rolling validation**.

- **Round 1:** parameters estimated on 2005–2022.
- **Round 2:** parameters re-estimated on 2005–2023, warm-started from round 1.

Each round forecasts 12 months ahead from its December origin.

In [2]:
# ============================================================
# 1. SARIMAX (per grid, orders from rolling validation)
# ============================================================

from joblib import Parallel, delayed
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.statespace.sarimax import SARIMAX


# ------------------------------------------------------------
# SARIMAX SETTINGS (same as rolling validation)
# ------------------------------------------------------------

EXOG_LAG = 12

SEASONAL_D = 1

MAX_ITER = 200

N_JOBS = -1


# ------------------------------------------------------------
# Orders selected during rolling validation
# ------------------------------------------------------------

if not SARIMAX_ORDERS_FILE.exists():

    raise FileNotFoundError(
        f"{SARIMAX_ORDERS_FILE} not found. Run Standalone_Models.ipynb first."
    )

sarimax_orders = (
    pd.read_csv(SARIMAX_ORDERS_FILE)
    .query("round == 1")
    .set_index("grid_number")
)

SARIMAX_ORDERS = {
    grid_number: (
        ast.literal_eval(row["order"]),
        ast.literal_eval(row["seasonal_order"])
    )
    for grid_number, row in sarimax_orders.iterrows()
}

missing_orders = set(GRID_INFO["grid_number"]) - set(SARIMAX_ORDERS)

if missing_orders:

    raise ValueError(f"No SARIMAX order for grids: {sorted(missing_orders)}")


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def fit_sarimax(endog, exog, order, seasonal_order, start_params=None):
    """Fit one SARIMAX model, silencing optimiser warnings."""

    with warnings.catch_warnings():

        warnings.simplefilter("ignore")

        return SARIMAX(
            endog,
            exog=exog,
            order=order,
            seasonal_order=seasonal_order
        ).fit(
            start_params=start_params,
            maxiter=MAX_ITER,
            disp=False
        )


def run_sarimax_grid(grid_number, frame, order, seasonal_order):

    series = (
        frame.set_index("date")
        .asfreq("MS")
    )

    exog = series[EXOG_COLS].shift(EXOG_LAG).iloc[EXOG_LAG:]

    endog = series[GHI_COL].iloc[EXOG_LAG:]


    predictions = []

    diagnostics = []

    result = None

    for rnd in ROUNDS.itertuples(index=False):

        result = fit_sarimax(
            endog.loc[:rnd.train_end_date],
            exog.loc[:rnd.train_end_date],
            order,
            seasonal_order,
            start_params=None if result is None else result.params
        )

        forecast = result.get_forecast(
            steps=HORIZON,
            exog=exog.loc[rnd.test_start:rnd.test_end]
        ).predicted_mean

        burn_in = order[1] + SEASONAL_D * SEASONAL_PERIOD

        ljung_box_p = acorr_ljungbox(
            result.resid.iloc[burn_in:],
            lags=[SEASONAL_PERIOD]
        )["lb_pvalue"].iloc[0]

        predictions.append(pd.DataFrame({
            "round":       rnd.round,
            "grid_number": grid_number,
            "target_date": forecast.index,
            "predicted":   forecast.to_numpy()
        }))

        diagnostics.append({
            "round":             rnd.round,
            "test_year":         rnd.test_year,
            "grid_number":       grid_number,
            "order":             str(order),
            "seasonal_order":    str(seasonal_order),
            "aic":               result.aic,
            "converged":         bool(result.mle_retvals.get("converged", True)),
            "ljung_box_p_lag12": ljung_box_p
        })

    return pd.concat(predictions, ignore_index=True), diagnostics


# ------------------------------------------------------------
# Run all grids in parallel
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("SARIMAX: FINAL TEST")
print("=" * 65)

sarimax_outputs = Parallel(n_jobs=N_JOBS, verbose=5)(
    delayed(run_sarimax_grid)(grid_number, frame, *SARIMAX_ORDERS[grid_number])
    for grid_number, frame in data.groupby("grid_number")
)

sarimax_diagnostics = pd.DataFrame(
    [row for output in sarimax_outputs for row in output[1]]
)

sarimax_diagnostics.to_csv(
    MODEL_DIR / "sarimax_final_test_diagnostics.csv",
    index=False
)

print(
    f"\nFits not converged: {(~sarimax_diagnostics['converged']).sum()} "
    f"of {len(sarimax_diagnostics)}"
)

sarimax_test = save_test_results(
    pd.concat([output[0] for output in sarimax_outputs], ignore_index=True),
    "sarimax",
    "SARIMAX"
)


SARIMAX: FINAL TEST


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  48 tasks      | elapsed:   20.2s
[Parallel(n_jobs=-1)]: Done  89 out of  93 | elapsed:   29.4s remaining:    1.2s



Fits not converged: 2 of 186

SARIMAX final test (2023-2024 pooled, 2232 forecasts): RMSE 0.3976   MAE 0.3140   nRMSE 7.71%   R² 0.7892
Saved: outputs\final_test\predictions\sarimax_final_test_predictions.csv


[Parallel(n_jobs=-1)]: Done  93 out of  93 | elapsed:   32.9s finished


## 2. LSTM

The same global multi-output LSTM as in rolling validation, retrained from scratch in each test round.

- **Architecture:** 2 × 64 LSTM units, dropout 0.2.
- **Training:** Adam (learning rate 1e-3), batch size 64, early stopping with patience 20. The last training year is used for early stopping.
- **Scaling:** the scaler is fitted on each round's training rows only.

In [3]:
# ============================================================
# 2. LSTM (global multi-output model, PyTorch)
# ============================================================

import copy

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler


# ------------------------------------------------------------
# LSTM SETTINGS (same as rolling validation)
# ------------------------------------------------------------

HIDDEN_SIZE = 64

NUM_LAYERS = 2

DROPOUT = 0.2

LEARNING_RATE = 1e-3

BATCH_SIZE = 64

MAX_EPOCHS = 200

PATIENCE = 20

SCALED_COLS = VARIABLES + STATIC_COLS

LSTM_FEATURES = SCALED_COLS + CALENDAR_COLS

GHI_INDEX = SCALED_COLS.index(GHI_COL)

torch.use_deterministic_algorithms(True)


# ------------------------------------------------------------
# Sequences: 12-month input window -> next 12 months
# ------------------------------------------------------------

def make_sequences(frame, feature_cols, window, horizon):
    """
    For every grid and origin month i, build
        X = features of months i-window+1 ... i
        Y = GHI of months i+1 ... i+horizon
    """

    X, Y, grid_numbers, origins = [], [], [], []

    for grid_number, group in frame.groupby("grid_number"):

        features = group[feature_cols].to_numpy(dtype=np.float32)

        target = group[GHI_COL].to_numpy(dtype=np.float32)

        dates = group["date"].to_numpy()

        for i in range(window - 1, len(group) - horizon):

            X.append(features[i - window + 1:i + 1])

            Y.append(target[i + 1:i + 1 + horizon])

            grid_numbers.append(grid_number)

            origins.append(dates[i])

    keys = pd.DataFrame({
        "grid_number": grid_numbers,
        "origin_date": pd.to_datetime(origins)
    })

    return np.stack(X), np.stack(Y), keys


# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

class LSTMForecaster(nn.Module):
    """Stacked LSTM with a multi-output linear head."""

    def __init__(self, n_features, hidden_size, num_layers, dropout, horizon):

        super().__init__()

        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True
        )

        self.dropout = nn.Dropout(dropout)

        self.output = nn.Linear(hidden_size, horizon)

    def forward(self, x):

        sequence_output, _ = self.lstm(x)

        return self.output(self.dropout(sequence_output[:, -1, :]))


def train_lstm(X_fit, Y_fit, X_es, Y_es, round_number):
    """Train with early stopping; return the best model and history."""

    set_seed(SEED)

    torch.manual_seed(SEED)

    model = LSTMForecaster(
        n_features=X_fit.shape[2],
        hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
        horizon=HORIZON
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    loss_function = nn.MSELoss()

    loader = DataLoader(
        TensorDataset(torch.from_numpy(X_fit), torch.from_numpy(Y_fit)),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED)
    )

    X_es = torch.from_numpy(X_es)

    Y_es = torch.from_numpy(Y_es)


    best_loss = np.inf

    best_state = None

    best_epoch = 0

    wait = 0

    history = []

    for epoch in range(1, MAX_EPOCHS + 1):

        model.train()

        running_loss = 0.0

        for X_batch, Y_batch in loader:

            optimizer.zero_grad()

            loss = loss_function(model(X_batch), Y_batch)

            loss.backward()

            optimizer.step()

            running_loss += loss.item() * len(X_batch)

        model.eval()

        with torch.no_grad():
            es_loss = loss_function(model(X_es), Y_es).item()

        history.append({
            "round": round_number,
            "epoch": epoch,
            "train_mse": running_loss / len(X_fit),
            "early_stop_mse": es_loss
        })

        if es_loss < best_loss:

            best_loss = es_loss

            best_state = copy.deepcopy(model.state_dict())

            best_epoch = epoch

            wait = 0

        else:

            wait += 1

            if wait >= PATIENCE:
                break

    model.load_state_dict(best_state)

    model.eval()

    return model, best_epoch, best_loss, history


# ------------------------------------------------------------
# Test rounds
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("LSTM: FINAL TEST")
print("=" * 65)

lstm_predictions = []

lstm_history = []

for rnd in ROUNDS.itertuples(index=False):

    frame = data[data["date"] <= rnd.test_end]

    scaler = MinMaxScaler().fit(
        frame.loc[frame["date"] <= rnd.train_end_date, SCALED_COLS]
    )

    scaled = frame.copy()

    scaled[SCALED_COLS] = scaler.transform(frame[SCALED_COLS])

    X, Y, keys = make_sequences(scaled, LSTM_FEATURES, WINDOW, HORIZON)


    last_target = keys["origin_date"] + pd.DateOffset(months=HORIZON)

    early_stop_start = rnd.train_end_date - pd.DateOffset(months=11)

    fit_mask = (last_target < early_stop_start).to_numpy()

    es_mask = (
        (last_target >= early_stop_start)
        & (last_target <= rnd.train_end_date)
    ).to_numpy()

    test_mask = (keys["origin_date"] == rnd.train_end_date).to_numpy()


    model, best_epoch, best_loss, history = train_lstm(
        X[fit_mask], Y[fit_mask],
        X[es_mask], Y[es_mask],
        rnd.round
    )

    lstm_history.extend(history)


    with torch.no_grad():
        predicted_scaled = model(torch.from_numpy(X[test_mask])).numpy()

    predicted = (
        predicted_scaled * scaler.data_range_[GHI_INDEX]
        + scaler.data_min_[GHI_INDEX]
    )

    test_grids = keys.loc[test_mask, "grid_number"].to_numpy()

    target_dates = pd.date_range(rnd.test_start, periods=HORIZON, freq="MS")

    lstm_predictions.append(pd.DataFrame({
        "round":       rnd.round,
        "grid_number": np.repeat(test_grids, HORIZON),
        "target_date": np.tile(target_dates, len(test_grids)),
        "predicted":   predicted.ravel()
    }))

    torch.save(
        {
            "state_dict": model.state_dict(),
            "features": LSTM_FEATURES,
            "scaled_cols": SCALED_COLS,
            "scaler_min": scaler.data_min_.tolist(),
            "scaler_max": scaler.data_max_.tolist(),
            "config": {
                "round": rnd.round,
                "train_end": rnd.train_end,
                "window": WINDOW,
                "horizon": HORIZON,
                "hidden_size": HIDDEN_SIZE,
                "num_layers": NUM_LAYERS,
                "dropout": DROPOUT,
                "learning_rate": LEARNING_RATE,
                "batch_size": BATCH_SIZE,
                "best_epoch": best_epoch,
                "seed": SEED
            }
        },
        MODEL_DIR / f"lstm_test_round{rnd.round}.pt"
    )

    print(
        f"Round {rnd.round} (test {rnd.test_year}): "
        f"fit {fit_mask.sum()}  early-stop {es_mask.sum()}  "
        f"best epoch {best_epoch}  early-stop MSE {best_loss:.5f}"
    )


pd.DataFrame(lstm_history).to_csv(
    MODEL_DIR / "lstm_final_test_training_history.csv",
    index=False
)

lstm_test = save_test_results(
    pd.concat(lstm_predictions, ignore_index=True),
    "lstm",
    "LSTM"
)


LSTM: FINAL TEST
Round 1 (test 2023): fit 16833  early-stop 1116  best epoch 15  early-stop MSE 0.00589
Round 2 (test 2024): fit 17949  early-stop 1116  best epoch 33  early-stop MSE 0.00502

LSTM final test (2023-2024 pooled, 2232 forecasts): RMSE 0.4162   MAE 0.3334   nRMSE 8.07%   R² 0.7691
Saved: outputs\final_test\predictions\lstm_final_test_predictions.csv


## 3. XGBoost

The same global direct multi-horizon XGBoost as in rolling validation, retrained in each test round with identical hyperparameters. The last training year is used for early stopping.

In [4]:
# ============================================================
# 3. XGBOOST (global direct multi-horizon model)
# ============================================================

from numpy.lib.stride_tricks import sliding_window_view

import xgboost as xgb


# ------------------------------------------------------------
# XGBOOST SETTINGS (same as rolling validation)
# ------------------------------------------------------------

XGB_PARAMS = {
    "n_estimators":          3000,
    "learning_rate":         0.05,
    "max_depth":             4,
    "min_child_weight":      1,
    "subsample":             0.8,
    "colsample_bytree":      0.8,
    "reg_lambda":            1.0,
    "objective":             "reg:squarederror",
    "tree_method":           "hist",
    "eval_metric":           "rmse",
    "early_stopping_rounds": 100,
    "random_state":          SEED,
    "n_jobs":                -1
}


LAG_COLS = [
    f"{variable}_lag{lag}"
    for variable in VARIABLES
    for lag in range(WINDOW)
]

XGB_FEATURES = (
    LAG_COLS
    + ["horizon", "target_month_sin", "target_month_cos"]
    + STATIC_COLS
)


# ------------------------------------------------------------
# Direct multi-horizon samples
# ------------------------------------------------------------

def make_direct_samples(frame, window, horizon):
    """
    One row per grid, origin month and horizon h:
        features = lags 0..window-1 before the origin
        target   = GHI at origin + h
    """

    tables = []

    for grid_number, group in frame.groupby("grid_number"):

        n = len(group)

        dates = group["date"].to_numpy()

        ghi = group[GHI_COL].to_numpy()

        lags = np.hstack([
            sliding_window_view(group[variable].to_numpy(), window)[:, ::-1]
            for variable in VARIABLES
        ])

        origin_index = np.arange(window - 1, n)

        for h in range(1, horizon + 1):

            valid = origin_index + h < n

            origins = origin_index[valid]

            table = pd.DataFrame(lags[valid], columns=LAG_COLS)

            table.insert(0, "grid_number", grid_number)

            table["origin_date"] = dates[origins]

            table["target_date"] = dates[origins + h]

            table["horizon"] = h

            for column in STATIC_COLS:
                table[column] = group[column].iloc[0]

            table["target"] = ghi[origins + h]

            tables.append(table)

    samples = pd.concat(tables, ignore_index=True)

    target_month = samples["target_date"].dt.month

    samples["target_month_sin"] = np.sin(2 * np.pi * target_month / 12)

    samples["target_month_cos"] = np.cos(2 * np.pi * target_month / 12)

    return samples


samples = make_direct_samples(data, WINDOW, HORIZON)

print("\n" + "=" * 65)
print("XGBOOST: FINAL TEST")
print("=" * 65)


# ------------------------------------------------------------
# Test rounds
# ------------------------------------------------------------

xgb_predictions = []

for rnd in ROUNDS.itertuples(index=False):

    early_stop_start = rnd.train_end_date - pd.DateOffset(months=11)

    fit_rows = samples["target_date"] < early_stop_start

    es_rows = (
        (samples["target_date"] >= early_stop_start)
        & (samples["target_date"] <= rnd.train_end_date)
    )

    test_rows = samples["origin_date"] == rnd.train_end_date


    model = xgb.XGBRegressor(**XGB_PARAMS)

    model.fit(
        samples.loc[fit_rows, XGB_FEATURES],
        samples.loc[fit_rows, "target"],
        eval_set=[(samples.loc[es_rows, XGB_FEATURES], samples.loc[es_rows, "target"])],
        verbose=False
    )

    predictions = samples.loc[test_rows, ["grid_number", "target_date"]].copy()

    predictions["round"] = rnd.round

    predictions["predicted"] = model.predict(samples.loc[test_rows, XGB_FEATURES])

    xgb_predictions.append(predictions)

    model.save_model(MODEL_DIR / f"xgboost_test_round{rnd.round}.json")

    print(
        f"Round {rnd.round} (test {rnd.test_year}): "
        f"fit {fit_rows.sum()}  early-stop {es_rows.sum()}  "
        f"best iteration {model.best_iteration}  early-stop RMSE {model.best_score:.5f}"
    )


xgb_test = save_test_results(
    pd.concat(xgb_predictions, ignore_index=True),
    "xgboost",
    "XGBoost"
)


XGBOOST: FINAL TEST
Round 1 (test 2023): fit 208134  early-stop 13392  best iteration 64  early-stop RMSE 0.43358
Round 2 (test 2024): fit 221526  early-stop 13392  best iteration 1606  early-stop RMSE 0.36146

XGBoost final test (2023-2024 pooled, 2232 forecasts): RMSE 0.4055   MAE 0.3174   nRMSE 7.87%   R² 0.7808
Saved: outputs\final_test\predictions\xgboost_final_test_predictions.csv


## 4. Reference forecasts and final comparison

- **Climatology:** the mean GHI of each calendar month over the round's training years.
- **Seasonal naive:** the GHI of the same month in the previous year.

All metrics are **pooled over both test years** (2023 + 2024, 2,232 forecasts per model). **Skill** = `1 − RMSE_model / RMSE_climatology`.

In [5]:
# ============================================================
# 4. REFERENCE FORECASTS AND FINAL COMPARISON
# ============================================================

# ------------------------------------------------------------
# Climatology and seasonal naive
# ------------------------------------------------------------

climatology_predictions = []

naive_predictions = []

for rnd in ROUNDS.itertuples(index=False):

    train = data[data["date"] <= rnd.train_end_date]

    targets = data.loc[
        (data["date"] >= rnd.test_start) & (data["date"] <= rnd.test_end),
        ["grid_number", "date", "month"]
    ].rename(columns={"date": "target_date"})

    monthly_mean = (
        train.groupby(["grid_number", "month"], as_index=False)[GHI_COL]
        .mean()
        .rename(columns={GHI_COL: "predicted"})
    )

    previous_year = (
        train.loc[train["date"].dt.year == rnd.test_year - 1, ["grid_number", "month", GHI_COL]]
        .rename(columns={GHI_COL: "predicted"})
    )

    climatology_predictions.append(
        targets.merge(monthly_mean, on=["grid_number", "month"]).assign(round=rnd.round)
    )

    naive_predictions.append(
        targets.merge(previous_year, on=["grid_number", "month"]).assign(round=rnd.round)
    )


save_test_results(
    pd.concat(climatology_predictions, ignore_index=True),
    "climatology",
    "Climatology"
)

save_test_results(
    pd.concat(naive_predictions, ignore_index=True),
    "seasonal_naive",
    "Seasonal naive"
)


# ------------------------------------------------------------
# Combine every model that has been run
# ------------------------------------------------------------

MODELS = {
    "climatology":    "Climatology",
    "seasonal_naive": "Seasonal naive",
    "sarimax":        "SARIMAX",
    "lstm":           "LSTM",
    "xgboost":        "XGBoost"
}

all_predictions = []

for name, label in MODELS.items():

    path = PREDICTION_DIR / f"{name}_final_test_predictions.csv"

    if path.exists():
        all_predictions.append(pd.read_csv(path, parse_dates=["target_date"]))
    else:
        print(f"Skipping {label}: run its section first")

all_predictions = pd.concat(all_predictions, ignore_index=True)

model_order = [label for label in MODELS.values() if label in set(all_predictions["model"])]


# ------------------------------------------------------------
# Final metrics: one row per model, 2023-2024 pooled
# ------------------------------------------------------------

final_metrics = (
    pooled_metrics(all_predictions, ["model"])
    .set_index("model")
    .reindex(model_order)
)

final_metrics["skill_vs_climatology"] = (
    1 - final_metrics["RMSE"] / final_metrics.loc["Climatology", "RMSE"]
)

final_metrics.insert(0, "test_period", f"{TEST_YEARS[0]}-{TEST_YEARS[-1]}")

final_metrics.to_csv(OUTPUT_DIR / "final_test_metrics.csv")


# ------------------------------------------------------------
# Per-grid metrics, pooled over the 24 test months
# ------------------------------------------------------------

final_by_grid = pooled_metrics(all_predictions, ["model"] + GRID_COLS)

climatology_rmse = (
    final_by_grid[final_by_grid["model"] == "Climatology"]
    .set_index("grid_number")["RMSE"]
)

final_by_grid["skill_vs_climatology"] = (
    1 - final_by_grid["RMSE"] / final_by_grid["grid_number"].map(climatology_rmse)
)

final_by_grid.insert(1, "test_period", f"{TEST_YEARS[0]}-{TEST_YEARS[-1]}")

final_by_grid.to_csv(OUTPUT_DIR / "final_test_metrics_by_grid.csv", index=False)


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("\n" + "=" * 65)
print(
    f"FINAL TEST RESULTS ({TEST_YEARS[0]}-{TEST_YEARS[-1]} pooled, year-ahead, "
    f"{N_GRIDS} grids)"
)
print("=" * 65)

print(
    final_metrics[["RMSE", "MAE", "nRMSE_pct", "R2", "skill_vs_climatology", "n_forecasts"]]
    .round(4)
    .to_string()
)

print("\nPer-grid summary (metrics pooled over 24 months per grid):")

print(
    final_by_grid.groupby("model", sort=False)
    .agg(
        median_RMSE=("RMSE", "median"),
        median_R2=("R2", "median"),
        grids_beating_climatology=("skill_vs_climatology", lambda s: int((s > 0).sum()))
    )
    .reindex(model_order)
    .round(4)
    .to_string()
)

print(f"\nSaved: {OUTPUT_DIR / 'final_test_metrics.csv'}")
print(f"Saved: {OUTPUT_DIR / 'final_test_metrics_by_grid.csv'}")


Climatology final test (2023-2024 pooled, 2232 forecasts): RMSE 0.4074   MAE 0.3250   nRMSE 7.90%   R² 0.7788
Saved: outputs\final_test\predictions\climatology_final_test_predictions.csv

Seasonal naive final test (2023-2024 pooled, 2232 forecasts): RMSE 0.6016   MAE 0.4832   nRMSE 11.67%   R² 0.5176
Saved: outputs\final_test\predictions\seasonal_naive_final_test_predictions.csv

FINAL TEST RESULTS (2023-2024 pooled, year-ahead, 93 grids)
                  RMSE     MAE  nRMSE_pct      R2  skill_vs_climatology  n_forecasts
model                                                                               
Climatology     0.4074  0.3250     7.9026  0.7788                0.0000         2232
Seasonal naive  0.6016  0.4832    11.6708  0.5176               -0.4768         2232
SARIMAX         0.3976  0.3140     7.7140  0.7892                0.0239         2232
LSTM            0.4162  0.3334     8.0749  0.7691               -0.0218         2232
XGBoost         0.4055  0.3174     7.8669  0.7

## References

The same libraries and methods as `Standalone_Models.ipynb`; see that notebook's reference list. Additional reference for the evaluation metrics:

- Hyndman, R. J., & Koehler, A. B. (2006). Another look at measures of forecast accuracy. *International Journal of Forecasting, 22*(4), 679–688. https://doi.org/10.1016/j.ijforecast.2006.03.001
- Chicco, D., Warrens, M. J., & Jurman, G. (2021). The coefficient of determination R-squared is more informative than SMAPE, MAE, MAPE, MSE and RMSE in regression analysis evaluation. *PeerJ Computer Science, 7*, e623. https://doi.org/10.7717/peerj-cs.623